# OntologyRAG-Q — best configuration + Chain-of-Thought, run with Gemma

Runs the best-performing setup from Table 4 of Al-Azani et al. (EMNLP 2025):

**Ayat-Ontology chunking · k = 6 · Similarity · temperature = 0.0 · multilingual-e5-small**

with Gemma as the generator, and adds **chain-of-thought (CoT) reasoning** on top of it.

### What the chain-of-thought adds

Retrieval is left exactly as the paper's best row — same chunking, same embedder, same
k = 6, same cosine similarity, same greedy decoding. The only thing that changes is the
*generation* stage:

1. The k retrieved passages are numbered `[المقطع 1] … [المقطع 6]` so the model can
   refer to them while reasoning.
2. The model is asked to write its reasoning under `التفكير:` — identify what the
   question asks, scan each numbered passage, pick the ones that actually contain the
   answer, pull out the supporting sentences with the Ayah quoted verbatim, and check
   every claim is grounded (otherwise answer `لا أعرف`).
3. It then writes the answer under `الإجابة النهائية:`.
4. Only the text after `الإجابة النهائية:` is scored. The reasoning is stripped out and
   saved separately, so BLEU / CHRF / BERTScore stay comparable with the paper's numbers.

This keeps the experiment honest: any change in the metrics comes from the reasoning
step, not from a different retriever.

### This run answers **2000 questions** (`N_EVAL_SAMPLES = 2000`).

That is 2000 of the 2,350 Direct questions — a real evaluation, not a sample check. Budget
for it: roughly **30 GPU hours**, which is more than one Kaggle session (12 h) and about a
whole week's GPU quota. Two things make that survivable:

- Answers are saved to `results/preds_..._n2000_seed42.json` after **every single
  question**, and the run skips anything already in that file. So stopping is safe.
- But the file has to still be there next time. Turn on **Settings → Persistence →
  Variables and Files**, or download the predictions file before the session ends.
  Without that, `/kaggle/working` is wiped and you restart from zero.

If you only want a quick answer on whether CoT helps, `N_EVAL_SAMPLES = 200` costs about
3 hours and is already enough to see a real difference.

### Comparing CoT against the paper's prompt

Run once with `USE_COT = True`, then set `USE_COT = False` and Run All again. Both runs
use the same seed, so they answer the *same* questions, and the last cell prints the two
rows side by side with the deltas. Predictions and results are written to separate files
per mode, so neither run overwrites the other.

Note this doubles the cost — 2000 questions twice is ~60 GPU hours. A cheaper route is to
run the baseline at a smaller `n` first to see the direction, then spend the big budget on
one mode only. Comparisons are only valid between runs with the **same** `n`.

---

### Setup on Kaggle (do this first)

1. Go to https://huggingface.co/google/gemma-3-4b-it and click **Acknowledge license**.
2. Go to https://huggingface.co/settings/tokens and create a token, type **Read**. Copy it.
3. In this notebook: **Add-ons → Secrets → Add a new secret**. Label it `HF_TOKEN`, paste the token, tick the checkbox.
4. **Settings → Accelerator → GPU**.
5. Use the **same Hugging Face account** for steps 1 and 2.

### How to run

Leave `QUICK_TEST = True` and press **Run All**. This takes about 30 minutes (most of it
building the search index once) and answers 5 questions, just to prove everything works.

Then set `QUICK_TEST = False` and Run All again. The index is cached, so it goes straight
to answering the evaluation questions.

If it stops early, just Run All again — it continues from where it stopped.

In [ ]:
# ================= CONFIG =================
QUICK_TEST = False                      # True = 5 questions. Set False for the real run.

# ---- chain-of-thought ----
USE_COT = True                          # True = reason step by step, then answer.
                                        # False = the paper's original single-shot prompt.

# The CoT answer is the text after 'الإجابة النهائية:'. Without a length instruction the
# model tends to dump its whole reasoning into that section, which inflates the answer
# and costs BLEU against the short reference answers. This asks for one short paragraph.
# Turn it off if you want the reasoning step measured with no length pressure at all.
CONCISE_FINAL_ANSWER = True

SHOW_TRACES = 2                         # how many full reasoning traces to print at the end

MODEL_ID = "google/gemma-3-4b-it"       # or "google/gemma-3-1b-it"
QUANT_BITS = 4                          # 8 = better quality, may not fit on a T4

N_EVAL_SAMPLES = 2000                   # None = all 2,350 Direct questions
SEED = 42

# Which books to search.
#   "all"    -- all 15 Tafsir books (55,471 chunks). Harder search.
#   "source" -- only the 2 books the answers were written from (~7,500 chunks).
#
# The paper's Limitations says "we only performed the analysis using one
# source", which is ambiguous. If they searched one book, their search was
# far easier than searching all 15, which would explain part of their high
# scores. Running both settings tells you how much the corpus size matters.
CORPUS = "all"

# The paper's best row -- do not change these. CoT changes the generator only.
TOP_K = 6
EMBED_MODEL_ID = "intfloat/multilingual-e5-small"

MAX_CHARS_PER_CHUNK = 2000
MAX_PROMPT_TOKENS = 8192
# Reasoning is generated before the answer, so CoT needs a bigger budget -- with 512 the
# model can run out of tokens mid-reasoning and never reach 'الإجابة النهائية:'.
# Raising the cap is nearly free: generation time depends on the tokens actually
# produced, and the model stops on its own when it is done. The cap only costs
# anything on the answers that would otherwise have been cut off mid-sentence.
MAX_NEW_TOKENS = 1024 if USE_COT else 512
COT_RETRY_NEW_TOKENS = 1536             # one retry, only when the first run hit the cap
BERTSCORE_MODEL = "bert-base-multilingual-cased"
RESULTS_DIR = "results"

MODE = "cot" if USE_COT else "base"

if QUICK_TEST:
    N_EVAL_SAMPLES = 5
    print("QUICK_TEST on: 5 questions only.")
print(f"{MODEL_ID} | {QUANT_BITS}-bit | k={TOP_K} | corpus={CORPUS} "
      f"| n={N_EVAL_SAMPLES} | mode={MODE}")

# A big run will not finish inside one Kaggle session, so say so up front rather than
# letting it die at the 12-hour limit with no warning.
if N_EVAL_SAMPLES is None or N_EVAL_SAMPLES > 200:
    n_show = N_EVAL_SAMPLES or 2350
    print(f"\nHeads up: {n_show} questions at roughly 45-75 s each is about "
          f"{n_show * 60 / 3600:.0f} GPU hours.")
    print("Kaggle stops a session at 12 h, so this needs several sittings. The run "
          "resumes from")
    print(f"  {RESULTS_DIR}/preds_*_{MODE}_n{N_EVAL_SAMPLES}_seed{SEED}.json")
    print("but only if that file survives. Turn on Settings > Persistence "
          "(Variables and Files),")
    print("or download the file before each session ends -- otherwise every "
          "session starts from zero.")

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.makedirs(RESULTS_DIR, exist_ok=True)

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded.")
except Exception as e:
    print(f"Could not read HF_TOKEN: {type(e).__name__}")
    print("Fix: Add-ons > Secrets > new secret labelled exactly HF_TOKEN, checkbox ticked.")

!pip install -q -U transformers accelerate bitsandbytes faiss-cpu sentence-transformers openpyxl sacrebleu bert-score tqdm

In [ ]:
# ========== CHECK EVERYTHING BEFORE THE SLOW STEPS ==========
import torch
ok = True

if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"[OK]   GPU: {p.name}, {p.total_memory/1e9:.1f} GB")
else:
    ok = False
    print("[FAIL] No GPU. Settings > Accelerator > GPU, then restart the session.")

token = os.environ.get("HF_TOKEN")
if not token:
    ok = False
    print("[FAIL] HF_TOKEN missing -- see the cell above.")
else:
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    try:
        print(f"[OK]   Token belongs to: {api.whoami().get('name')}")
    except Exception:
        ok = False
        print("[FAIL] Token rejected -- create a fresh token of type 'Read'.")
    try:
        api.model_info(MODEL_ID)
        print(f"[OK]   Access to {MODEL_ID} confirmed.")
    except Exception:
        ok = False
        print(f"[FAIL] No access to {MODEL_ID}.")
        print(f"       Open https://huggingface.co/{MODEL_ID}, click 'Acknowledge license',")
        print("       logged in as the user printed above.")

print("\nReady to run." if ok else "\nFix the [FAIL] items before continuing.")

In [ ]:
# ========== LOAD THE MODEL ==========
# Gemma-3-4B is a multimodal checkpoint and needs Gemma3ForConditionalGeneration.
# Gemma-3-1B is text-only and needs a plain causal-LM class. Try each in turn.
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM

bnb = (BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                          bnb_4bit_quant_type="nf4") if QUANT_BITS == 4 else
       BitsAndBytesConfig(load_in_8bit=True) if QUANT_BITS == 8 else None)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

classes = []
for name in ["Gemma3ForConditionalGeneration", "AutoModelForImageTextToText"]:
    try:
        classes.append(getattr(__import__("transformers", fromlist=[name]), name))
    except (ImportError, AttributeError):
        pass
classes.append(AutoModelForCausalLM)

model, errs = None, []
for cls in classes:
    try:
        model = cls.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
        print(f"Loaded with {cls.__name__} ({QUANT_BITS}-bit).")
        break
    except Exception as e:
        errs.append(f"{cls.__name__}: {type(e).__name__}: {str(e)[:200]}")

if model is None:
    for e in errs:
        print(" -", e)
    raise RuntimeError("Could not load the model -- see errors above.")
model.eval()

# Some chat templates accept a separate system turn, some do not. Detect it.
try:
    tokenizer.apply_chat_template([{"role": "system", "content": "x"},
                                   {"role": "user", "content": "y"}],
                                  tokenize=False, add_generation_prompt=True)
    SUPPORTS_SYSTEM_ROLE = True
except Exception:
    SUPPORTS_SYSTEM_ROLE = False
print(f"System role supported: {SUPPORTS_SYSTEM_ROLE}")

In [ ]:
# ========== DOWNLOAD THE DATA ==========
import json, requests
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

BASE = "https://raw.githubusercontent.com/sazani/OntologyRAG-Q/main"
ALL_BOOKS = [
    "aashoor_v2.xlsx", "alaloosi_v2.xlsx", "almawirdee_v2.xlsx", "almuyassar_v2.xlsx",
    "alrazi_v2.xlsx", "altasheel_v2.xlsx", "aysaraAltafasir_v2.xlsx", "fathAlqadeer_v2.xlsx",
    "fathaAlbayan_v2.xlsx", "katheer_v2.xlsx", "mukhtasar_v2.xlsx", "qurtubi_v2.xlsx",
    "saadi_v2.xlsx", "tabari_v2.xlsx", "zadAlmaseer_v2.xlsx",
]
# The two books the reference answers were written from.
SOURCE_BOOKS = ["aysaraAltafasir_v2.xlsx", "almuyassar_v2.xlsx"]

BOOK_FILES = ALL_BOOKS if CORPUS == "all" else SOURCE_BOOKS
CHUNKS_PATH = f"chunks_{CORPUS}.json"
INDEX_PATH = f"index_{CORPUS}.faiss"

os.makedirs("books", exist_ok=True)
for fname in BOOK_FILES:
    p = os.path.join("books", fname)
    if not os.path.exists(p):
        r = requests.get(f"{BASE}/Resources/Tafaser/Tafaser_DS2/{fname}")
        r.raise_for_status()
        open(p, "wb").write(r.content)

if not os.path.exists("OntologyQA_v1.json"):
    r = requests.get(f"{BASE}/Resources/OntologyQA_v1.json")
    r.raise_for_status()
    open("OntologyQA_v1.json", "wb").write(r.content)

with open("OntologyQA_v1.json", encoding="utf-8") as f:
    qa_data = json.load(f)

chunks = index = embed_model = None
if os.path.exists(CHUNKS_PATH) and os.path.exists(INDEX_PATH):
    with open(CHUNKS_PATH, encoding="utf-8") as f:
        chunks = json.load(f)
    index = faiss.read_index(INDEX_PATH)
    embed_model = SentenceTransformer(EMBED_MODEL_ID)
    print(f"Cached: {len(chunks)} chunks, {index.ntotal} vectors.")
else:
    print(f"{len(BOOK_FILES)} books ready, {len(qa_data)} QA pairs. Building index below.")

In [ ]:
# ========== AYAT-ONTOLOGY CHUNKING ==========
# One chunk per verse (or verse range), with the ontology fields -- surah name,
# surah number, verse range -- written into the chunk text, as the paper describes.
#
# NOTE: aysaraAltafasir_v2.xlsx uses different column names from the other 14
# books. Handling only the majority schema silently drops all 1,290 of its rows,
# and that book is the source of ~90% of the Direct questions' answers. Both
# schemas are handled here. With CORPUS="all" this gives 55,471 chunks, which
# is exactly the total reported in the paper's Table 6.
import pandas as pd

STANDARD = {"surah": "SURA_num", "start": "Verse_Number_start",
            "end": "Verse_Number_end", "verse": "passages", "tafsir": "Tafsir"}
AYSARA = {"surah": "SURA_num", "aya_range": "AYA_num",
          "verse": "Ayah", "tafsir": "Tafsir"}
SCHEMAS = {"aysaraAltafasir_v2.xlsx": AYSARA}


def _parse_aya_range(v):
    """'217-218' -> (217, 218); '14-15-16' -> (14, 16); '7' -> (7, 7)."""
    nums = [int(p) for p in str(v).split("-") if p.strip().isdigit()]
    return (min(nums), max(nums)) if nums else (None, None)


if chunks is None:
    surah_names = {}
    for d in qa_data:
        sid, nm = d.get("Sura_ID"), d.get("SURA_name")
        if sid is not None and nm:
            try:
                surah_names[int(sid)] = nm
            except (ValueError, TypeError):
                pass

    chunks, per_book = [], {}
    for fname in BOOK_FILES:
        src = fname.replace("_v2.xlsx", "")
        sc = SCHEMAS.get(fname, STANDARD)
        df = pd.read_excel(os.path.join("books", fname))
        made = 0

        for _, row in df.iterrows():
            su = row.get(sc["surah"])
            if pd.isna(su):
                continue
            if "aya_range" in sc:
                st, en = _parse_aya_range(row.get(sc["aya_range"]))
            else:
                st, en = row.get(sc["start"]), row.get(sc["end"])
            if st is None or en is None or pd.isna(st) or pd.isna(en):
                continue

            tafsir = str(row.get(sc["tafsir"], "")).strip()
            if not tafsir or tafsir == "nan":
                continue
            verse = str(row.get(sc["verse"], "")).strip()

            su, st, en = int(su), int(st), int(en)
            sname = surah_names.get(su, f"سورة {su}")
            chunks.append({
                "surah_number": su, "surah_name": sname,
                "start_ayah": st, "end_ayah": en, "source": src,
                "chunk_text": (f"سورة: {sname} (رقم {su})\n"
                               f"الآية رقم {st} إلى {en}: {verse}\n"
                               f"التفسير ({src}): {tafsir}"),
            })
            made += 1

        per_book[src] = made
        print(f"{src:20} {len(df):5} rows -> {made:5} chunks")

    empty = [b for b, m in per_book.items() if m == 0]
    if empty:
        raise RuntimeError(f"No chunks produced from {empty} -- column schema mismatch.")

    with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
        json.dump(chunks, f, ensure_ascii=False)

    print(f"\nTOTAL: {len(chunks)} chunks", end="  ")
    if CORPUS == "all":
        print("-- matches the paper's Table 6 total (55,471)." if len(chunks) == 55471
              else "-- expected 55,471 per the paper's Table 6; check before trusting results.")
    else:
        print("(source books only)")
else:
    print(f"Skipped -- {len(chunks)} chunks cached.")

In [ ]:
# ========== BUILD THE SEARCH INDEX (the slow step) ==========
# normalize_embeddings + IndexFlatIP together == cosine similarity,
# which is the "Similarity" search in the paper's best row.
if index is None:
    embed_model = SentenceTransformer(EMBED_MODEL_ID)
    # e5 models need "passage: " on stored text and "query: " on searches.
    texts = ["passage: " + c["chunk_text"] for c in chunks]
    print(f"Embedding {len(texts)} chunks. This is the slow part -- do not interrupt.")
    emb = np.array(embed_model.encode(texts, batch_size=64, show_progress_bar=True,
                                      normalize_embeddings=True), dtype="float32")
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    faiss.write_index(index, INDEX_PATH)
    print("Index built:", index.ntotal, "chunks. Cached for next time.")
else:
    print(f"Skipped -- {index.ntotal} vectors cached.")

In [ ]:
# ========== PROMPTS: PAPER BASELINE + CHAIN OF THOUGHT ==========
import re

# The baseline prompt is copied word for word from the paper, §4.
SYSTEM_PROMPT = """You are an expert in interpreting the Quran, specifically
designed to answer users' questions. Provide answers solely based on the
context provided below. Do not draw upon any external or prior knowledge or
information. If the answer is not found within the given context, respond
with 'I don't know.' Ensure that the Ayah (verses) are quoted verbatim as
they appear in the Quran. All answers should be provided in Arabic."""

# The CoT prompt keeps every constraint of the baseline word for word -- grounding,
# 'I don't know', verbatim Ayat, Arabic -- and only adds the reasoning procedure and a
# fixed two-section output format. The section marker is what lets the reasoning be
# stripped before scoring, so it has to be stated explicitly and never translated.
_COT_STEPS = """
Think step by step before you answer. Your response must contain exactly two
sections, with these two Arabic headings written exactly as shown, each on its own
line, and nothing at all before the first heading. Do not shorten 'الإجابة النهائية'
and do not replace it with any other word:

التفكير:
١. حدد بدقة ما يسأل عنه السؤال: السورة، ورقم الآية، والمفهوم المطلوب.
٢. افحص كل مقطع من مقاطع السياق المرقّمة، واذكر أرقام المقاطع التي تتضمن الإجابة
   فعليًا (استخدم أرقام المقاطع المعطاة فقط، من ١ إلى ٦)، وتجاهل ما لا يتعلق بالسؤال.
٣. اذكر بإيجاز ما يدعم الإجابة في تلك المقاطع.
٤. تحقق من أن كل جزء من إجابتك مذكور صراحة في السياق. إن لم يكن كذلك، فالإجابة
   هي 'لا أعرف'.

قيود هذا القسم: أربع نقاط، سطر واحد لكل نقطة. لا تنسخ نصوصًا طويلة من السياق هنا؛
النقل الحرفي للآيات يكون في الإجابة النهائية، لا في التفكير. القسم الأهم هو الإجابة
النهائية، فلا تستهلك المساحة في التفكير.

الإجابة النهائية:
{answer_rule}"""

_ANSWER_RULE_CONCISE = """اكتب هنا الإجابة النهائية بالعربية فقط، في فقرة واحدة موجزة
مكتفية بذاتها، بأسلوب كتب التفسير. لا تذكر أرقام المقاطع، ولا تشر إلى السياق أو إلى
خطوات التفكير، ولا تكرر السؤال."""

_ANSWER_RULE_PLAIN = """اكتب هنا الإجابة النهائية بالعربية فقط، مكتفية بذاتها. لا تذكر
أرقام المقاطع، ولا تشر إلى السياق أو إلى خطوات التفكير."""

SYSTEM_PROMPT_COT = SYSTEM_PROMPT + "\n" + _COT_STEPS.format(
    answer_rule=_ANSWER_RULE_CONCISE if CONCISE_FINAL_ANSWER else _ANSWER_RULE_PLAIN)

ACTIVE_SYSTEM_PROMPT = SYSTEM_PROMPT_COT if USE_COT else SYSTEM_PROMPT

# Matches 'الإجابة النهائية:' and the spellings the model actually drifts to --
# hamza-less 'الاجابة', 'الجواب النهائي', an English fallback, markdown bold or a
# heading around it, and a missing colon.
_STRICT_FINAL = re.compile(
    r"(?:^|\n)\s*(?:#{1,6}\s*)?(?:\*{1,3}|_{1,2})?\s*"
    r"(?:الإجابة\s*النهائية|الاجابة\s*النهائية|الإجابه\s*النهائيه|"
    r"الجواب\s*النهائي|Final\s*Answer)"
    r"\s*(?:\*{1,3}|_{1,2})?\s*[:：]?\s*")

# Gemma often shortens the heading to just 'الإجابة:' or 'الخلاصة:'. Those are only
# tried when the full heading is absent, and a colon is required, so an ordinary
# sentence containing the word 'الجواب' cannot be mistaken for a heading.
_LOOSE_FINAL = re.compile(
    r"(?:^|\n)\s*(?:#{1,6}\s*)?(?:\*{1,3}|_{1,2})?\s*"
    r"(?:الإجابة|الاجابة|الجواب|الخلاصة|Answer)"
    r"\s*(?:\*{1,3}|_{1,2})?\s*[:：]\s*")

# Lines that look like reasoning steps: '1.', '٢)', '- ', '* '.
_STEP_LINE = re.compile(r"^\s*(?:[*\-–]|[0-9٠-٩]{1,2}[\.\)])\s+", re.M)

_THINK_MARKER = re.compile(
    r"(?:^|\n)\s*(?:#{1,6}\s*)?(?:\*{1,3}|_{1,2})?\s*"
    r"(?:التفكير|التحليل|خطوات\s*التفكير|Reasoning|Thinking)"
    r"\s*(?:\*{1,3}|_{1,2})?\s*[:：]?\s*")


def _tidy(text):
    """Drop leftover markdown and stray leading punctuation from a parsed section."""
    text = re.sub(r"\*{2,3}|_{2,}|^#{1,6}\s*", "", text.strip())
    return re.sub(r"^[\s:：\-–—•\.]+", "", text).strip()


def _last_paragraph(text):
    paras = [p for p in re.split(r"\n\s*\n", text.strip()) if p.strip()]
    return _tidy(paras[-1]) if paras else ""


def split_cot(raw, truncated=False):
    """Split a CoT generation into (reasoning, final answer, parsed_ok, reason).

    Only the final answer is scored, so everything here is about not putting the
    wrong text in it. `reason` records which route was taken, so a long run can be
    audited afterwards instead of just counting failures.
    """
    # 1. The heading we asked for, then the shortened forms Gemma drifts to.
    for pat, reason in ((_STRICT_FINAL, "ok"), (_LOOSE_FINAL, "short_heading")):
        matches = list(pat.finditer(raw))
        if not matches:
            continue
        m = matches[-1]                      # last one: the model sometimes restates it
        reasoning, answer = raw[:m.start()], _tidy(raw[m.end():])
        tm = _THINK_MARKER.search(reasoning)
        if tm:
            reasoning = reasoning[tm.end():]
        if answer:
            return _tidy(reasoning), answer, True, reason
        # Heading written, then nothing -- cut off at exactly the wrong moment.
        return _tidy(reasoning), "", False, "cut_off_at_heading"

    # 2. No answer heading anywhere. Did it start reasoning at all?
    tm = _THINK_MARKER.search(raw)
    if tm is None:
        # It ignored the format. If the text is not a list of steps, it simply
        # answered -- which is the baseline behaviour and perfectly scorable, so keep
        # ALL of it. Taking only the last paragraph here would throw away half a
        # good answer.
        if len(_STEP_LINE.findall(raw)) < 2:
            return "", _tidy(raw), True, "answered_without_headings"
        return _tidy(raw), _last_paragraph(raw), False, "steps_without_headings"

    # 3. Reasoning started and the answer section never arrived.
    body = raw[tm.end():]
    return (_tidy(body), _last_paragraph(body), False,
            "ran_out_of_room" if truncated else "no_answer_section")


print("Chain-of-thought prompting:", "ON" if USE_COT else "OFF (paper baseline)")
print("-" * 72)
print(ACTIVE_SYSTEM_PROMPT)

In [ ]:
# ========== RETRIEVAL + GENERATION ==========
# Retrieval is untouched: same chunks, same e5-small embedder, same cosine similarity,
# same k=6 as the paper's best row. CoT changes the generator only.


def retrieve(question, k=TOP_K):
    qv = np.array(embed_model.encode(["query: " + question], normalize_embeddings=True),
                  dtype="float32")
    _, ids = index.search(qv, k)
    return [chunks[i] for i in ids[0]]


def _chat_text(user_prompt, system_prompt):
    msgs = ([{"role": "system", "content": system_prompt},
             {"role": "user", "content": user_prompt}] if SUPPORTS_SYSTEM_ROLE else
            [{"role": "user", "content": f"{system_prompt}\n\n{user_prompt}"}])
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


def _assemble(top, question, chars, cot):
    parts = []
    for i, c in enumerate(top, 1):
        t = c["chunk_text"]
        t = (t[:chars] + " ...") if len(t) > chars else t
        # Numbering the passages gives step 2 of the reasoning something to point at.
        # "i of n" is spelled out because the model otherwise cites invented numbers.
        parts.append(f"[المقطع {i} من {len(top)}]\n{t}" if cot else t)
    body = f"Context: {chr(10).join(parts)}\nQuestion: {question}\nAnswer:"
    return _chat_text(body, SYSTEM_PROMPT_COT if cot else SYSTEM_PROMPT)


def _fit(top, question, cot):
    """Shrink the CONTEXT until the prompt fits, so the question is never cut off --
    it sits after the context, so truncating the prompt would delete it."""
    for c in [MAX_CHARS_PER_CHUNK, 1200, 800, 500, 300, 150]:
        text = _assemble(top, question, c, cot)
        if len(tokenizer(text)["input_ids"]) <= MAX_PROMPT_TOKENS:
            return text, c
    return text, 150


def _run(text, max_new_tokens):
    """Returns (generated_text, truncated).

    truncated=True means generation stopped because it hit the cap, not because the
    model finished. That distinction decides whether a retry is worth a minute of GPU.
    """
    for _ in range(3):
        try:
            inputs = tokenizer(text, return_tensors="pt").to(model.device)
            torch.cuda.empty_cache()
            with torch.no_grad():
                # do_sample=False is greedy decoding -- the paper's temperature=0.0.
                out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                     do_sample=False)
            new = out[0][inputs["input_ids"].shape[1]:]
            gen = tokenizer.decode(new, skip_special_tokens=True).strip()
            return gen, len(new) >= max_new_tokens
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            max_new_tokens = max(256, max_new_tokens // 2)
            print(f"    [out of memory -- retrying with max_new_tokens={max_new_tokens}]")
    raise RuntimeError("Out of memory even at the smallest generation budget.")


def generate(top, question, cot=None):
    """Returns {prediction, reasoning, raw, parsed_ok, parse_reason}.

    prediction is what gets scored.
    """
    cot = USE_COT if cot is None else cot
    text, chars = _fit(top, question, cot)

    try:
        raw, truncated = _run(text, MAX_NEW_TOKENS)
    except RuntimeError:
        # Last resort: squeeze the context instead of the generation budget.
        text = _assemble(top, question, 150, cot)
        raw, truncated = _run(text, MAX_NEW_TOKENS)

    if not cot:
        return {"prediction": raw, "reasoning": "", "raw": raw,
                "parsed_ok": True, "parse_reason": "baseline"}

    reasoning, answer, parsed_ok, reason = split_cot(raw, truncated)

    # Retry ONLY when the model actually ran out of room. Decoding is greedy, so a
    # model that stopped on its own regenerates the identical text -- the retry would
    # burn a minute per question to reach the same conclusion, which over 2000
    # questions is hours of GPU for nothing.
    if not parsed_ok and truncated:
        raw2, trunc2 = _run(text, COT_RETRY_NEW_TOKENS)
        r2, a2, ok2, reason2 = split_cot(raw2, trunc2)
        if ok2 or (a2 and not answer):
            reasoning, answer, raw, parsed_ok, reason = r2, a2, raw2, ok2, reason2

    return {"prediction": answer, "reasoning": reasoning, "raw": raw,
            "parsed_ok": parsed_ok, "parse_reason": reason}


def ask(question, verbose=True):
    top = retrieve(question)
    if verbose:
        for i, c in enumerate(top, 1):
            print(f"[{i}] {c['source']} · {c['surah_name']} {c['start_ayah']}-{c['end_ayah']}")
    res = generate(top, question)
    res["sources"] = [f"{c['source']}:{c['surah_number']}:"
                      f"{c['start_ayah']}-{c['end_ayah']}" for c in top]
    if verbose:
        if res["reasoning"]:
            print("\n--- التفكير (not scored) ---\n" + res["reasoning"])
        print("\n--- الإجابة النهائية (scored) ---\n" + res["prediction"])
        print(f"\n[parse: {res['parse_reason']}]")
        if not res["parsed_ok"]:
            print("[warn] no usable answer heading -- fell back to the last paragraph.")
    return res


direct_questions = [d for d in qa_data if d.get("Task") == "Direct" and d.get("Answer")]
print(f"Direct questions available: {len(direct_questions)}\n")
print("--- one test question ---")
print("Q:", direct_questions[0]["Question"], "\n")
_ = ask(direct_questions[0]["Question"])

In [ ]:
# ========== ANSWER THE QUESTIONS ==========
# Saved after every question. If this stops early, just run it again --
# it continues from where it left off instead of starting over.
# The filename carries the mode, so a CoT run and a baseline run never overwrite
# each other, and the same seed makes them answer the identical questions.
import random
from collections import Counter
from tqdm.auto import tqdm

model_tag = MODEL_ID.split("/")[-1]
path = os.path.join(RESULTS_DIR,
                    f"preds_{model_tag}_{CORPUS}_{MODE}_n{N_EVAL_SAMPLES}_seed{SEED}.json")

random.seed(SEED)
sample = (list(direct_questions) if N_EVAL_SAMPLES is None else
          random.sample(direct_questions, min(N_EVAL_SAMPLES, len(direct_questions))))

done = {}
if os.path.exists(path):
    done = {r["question"]: r for r in json.load(open(path, encoding="utf-8"))}
    print(f"Resuming: {len(done)}/{len(sample)} already answered.")

records = []
for item in tqdm(sample, desc=f"answering ({MODE})"):
    q = item["Question"]
    if q in done:
        records.append(done[q])
        continue
    res = ask(q, verbose=False)
    records.append({"question": q, "reference": item["Answer"],
                    "prediction": res["prediction"], "reasoning": res["reasoning"],
                    "raw": res["raw"], "parsed_ok": res["parsed_ok"],
                    "parse_reason": res["parse_reason"],
                    "sources": res["sources"], "mode": MODE})
    with open(path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=1)

n_bad = sum(1 for r in records if not r.get("parsed_ok", True))
n_empty = sum(1 for r in records if not r["prediction"].strip())
print(f"\n{len(records)} answers saved to {path}")

if USE_COT:
    # Which route each answer took. Counting failures alone tells you nothing about
    # what to change; the reason tells you whether to give it more tokens or to
    # tighten the prompt.
    reasons = Counter(r.get("parse_reason", "unknown") for r in records)
    WHAT = {
        "ok": "clean 'الإجابة النهائية:' heading",
        "short_heading": "shortened heading, e.g. 'الإجابة:' -- parsed fine",
        "answered_without_headings": "ignored the format and just answered -- kept in full",
        "steps_without_headings": "wrote steps with no headings -- last paragraph used",
        "cut_off_at_heading": "stopped right after the heading -- no answer text",
        "ran_out_of_room": "hit the token cap mid-reasoning -- raise MAX_NEW_TOKENS",
        "no_answer_section": "finished without an answer section -- prompt problem",
    }
    print("\nHow each answer was parsed:")
    for k, v in reasons.most_common():
        flag = " " if k in ("ok", "short_heading", "answered_without_headings") else "!"
        print(f" {flag} {v:5}  {k:26} {WHAT.get(k, '')}")
    print(f"\nUsable: {len(records) - n_bad}/{len(records)}.")
    if reasons["ran_out_of_room"] or reasons["cut_off_at_heading"]:
        print("Raise MAX_NEW_TOKENS -- some answers are being cut off.")

if n_empty:
    print(f"[warn] {n_empty} empty predictions -- they will score 0.")

In [ ]:
# ========== THE RESULT ==========
import sacrebleu
from bert_score import score as bert_score

preds = [r["prediction"] for r in records]
refs = [r["reference"] for r in records]

bleu = sacrebleu.corpus_bleu(preds, [refs]).score
chrf = sacrebleu.corpus_chrf(preds, [refs]).score
P, R, F1 = bert_score(preds, refs, model_type=BERTSCORE_MODEL, verbose=False)
p, r, f1 = P.mean().item()*100, R.mean().item()*100, F1.mean().item()*100

avg_pred_len = sum(len(x.split()) for x in preds) / max(1, len(preds))
avg_ref_len = sum(len(x.split()) for x in refs) / max(1, len(refs))
idk = sum(1 for x in preds if "لا أعرف" in x or "لا اعرف" in x or "I don't know" in x)

label = "chain-of-thought" if USE_COT else "paper baseline prompt"
print("=" * 72)
print(f"  OntologyRAG-Q best configuration — {MODEL_ID} — {label}")
print("=" * 72)
print(f"  BLEU             {bleu:6.2f}")
print(f"  CHRF             {chrf:6.2f}")
print(f"  BERT Precision   {p:6.2f}")
print(f"  BERT Recall      {r:6.2f}")
print(f"  BERT F1          {f1:6.2f}")
print("=" * 72)
print(f"  {len(records)} questions · {QUANT_BITS}-bit · {len(chunks)} chunks "
      f"· corpus={CORPUS} · seed {SEED}")
print(f"  mode={MODE} · avg answer {avg_pred_len:.0f} words vs reference "
      f"{avg_ref_len:.0f} · 'لا أعرف' on {idk}/{len(preds)}")
print(f"  BERTScore model: {BERTSCORE_MODEL}")
print()

print("Table 4 row:")
print(f"{'LLM':<16}{'Chunk':<16}{'Parameters':<44}{'Embed':<10}"
      f"{'BLEU':>7}{'CHRF':>7}{'Prec':>7}{'Rec':>7}{'F1':>7}")
print("-" * 121)
params = "k:6, Similarity, temperature=0.0" + (", CoT" if USE_COT else "")
print(f"{model_tag:<16}{'Ayat ontology':<16}{params:<44}{'E5-small':<10}"
      f"{bleu:>7.2f}{chrf:>7.2f}{p:>7.2f}{r:>7.2f}{f1:>7.2f}")

result = {"model": MODEL_ID, "quant_bits": QUANT_BITS, "corpus": CORPUS,
          "mode": MODE, "use_cot": USE_COT, "concise_final_answer": CONCISE_FINAL_ANSWER,
          "n": len(records), "seed": SEED, "n_chunks": len(chunks), "top_k": TOP_K,
          "bertscore_model": BERTSCORE_MODEL,
          "avg_pred_words": avg_pred_len, "avg_ref_words": avg_ref_len,
          "n_idk": idk, "n_parse_fallback": n_bad,
          "parse_reasons": (dict(Counter(r.get("parse_reason", "unknown")
                                         for r in records)) if USE_COT else {}),
          "BLEU": bleu, "CHRF": chrf, "BERT_P": p, "BERT_R": r, "BERT_F1": f1}
RESULT_PATH = os.path.join(
    RESULTS_DIR, f"result_{model_tag}_{CORPUS}_{MODE}_n{len(records)}.json")
json.dump(result, open(RESULT_PATH, "w"), indent=2)
print(f"\nSaved to {RESULT_PATH}")

In [ ]:
# ========== CoT vs BASELINE ==========
# Fills in once both modes have been run at the same n (flip USE_COT and Run All again).
other = "base" if USE_COT else "cot"
other_path = os.path.join(
    RESULTS_DIR, f"result_{model_tag}_{CORPUS}_{other}_n{len(records)}.json")

if not os.path.exists(other_path):
    print(f"No {other} run found at n={len(records)}.")
    print(f"Set USE_COT = {not USE_COT} and Run All to produce it, then re-run this cell.")
else:
    o = json.load(open(other_path))
    rows = [("chain-of-thought", result if USE_COT else o),
            ("paper baseline", o if USE_COT else result)]
    keys = ["BLEU", "CHRF", "BERT_P", "BERT_R", "BERT_F1"]
    print(f"{'prompt':<20}" + "".join(f"{k:>10}" for k in keys) + f"{'words':>8}")
    print("-" * 78)
    for name, d in rows:
        print(f"{name:<20}" + "".join(f"{d[k]:>10.2f}" for k in keys)
              + f"{d['avg_pred_words']:>8.0f}")
    cot_d, base_d = rows[0][1], rows[1][1]
    print("-" * 78)
    print(f"{'CoT - baseline':<20}"
          + "".join(f"{cot_d[k] - base_d[k]:>+10.2f}" for k in keys)
          + f"{cot_d['avg_pred_words'] - base_d['avg_pred_words']:>+8.0f}")
    print(f"\nBoth rows: {len(records)} identical questions (seed {SEED}), "
          f"corpus={CORPUS}, k={TOP_K}, greedy decoding.")

In [ ]:
# ========== LOOK AT THE REASONING ==========
# The reasoning is never scored, but it is the only way to see *why* an answer went
# wrong -- retrieval missed the verse, or the right passage was retrieved and the model
# reasoned past it.
if USE_COT and SHOW_TRACES:
    for rec in records[:SHOW_TRACES]:
        print("=" * 78)
        print("Q:", rec["question"])
        print("\nretrieved:", ", ".join(rec.get("sources", [])))
        print("\n--- التفكير ---\n" + (rec.get("reasoning") or "(none)"))
        print("\n--- الإجابة النهائية (scored) ---\n" + rec["prediction"])
        print("\n--- reference ---\n" + rec["reference"])
        print()
else:
    print("Nothing to show (baseline mode, or SHOW_TRACES = 0).")